<a href="https://colab.research.google.com/github/Sankha-Kuruppu/Body-Fat-Percentage-Mesuring-Model/blob/main/Model_Training_Using_Classic_Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

base = '/content/drive/MyDrive/AI_Project_Body_Fat_Percentage_Mesuring'
df = pd.read_csv(f'{base}/Processed/clean_bodyfat_dataset.csv')

print(df.shape)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(5628, 11)


,SEQN,RIDAGEYR,RIAGENDR,BMXWT,BMXHT,BMXBMI,BMXWAIST,BMXARMC,BMXCALF,BMXTHICR,DXDTOPF
0,31128.0,11.0,2.0,40.1,151.6,17.45,62.8,21.7,29.3,39.5,22.90
1,31129.0,15.0,1.0,74.6,167.7,26.53,97.8,32.6,40.6,55.9,30.00
2,31131.0,44.0,2.0,75.2,156.0,30.90,96.0,35.8,36.6,53.7,41.90
3,31133.0,16.0,2.0,45.0,163.7,16.79,62.0,22.3,31.7,41.3,17.46
4,31137.0,14.0,2.0,79.9,169.6,27.78,89.0,31.8,41.1,60.4,41.26


In [2]:
from sklearn.model_selection import train_test_split

# SEQN is just an ID, not a predictor — drop it from the features
feature_cols = ['RIDAGEYR', 'RIAGENDR', 'BMXWT', 'BMXHT', 'BMXBMI',
                 'BMXWAIST', 'BMXARMC', 'BMXCALF', 'BMXTHICR']

X = df[feature_cols]
y = df['DXDTOPF']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(4502, 9) (1126, 9)


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

print("Both models trained.")

Both models trained.


In [4]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(model, name):
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f"{name}: MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.3f}")
    return preds

lin_preds = evaluate(lin_reg, "Linear Regression")
rf_preds = evaluate(rf, "Random Forest")

Linear Regression: MAE=3.23  RMSE=4.03  R²=0.804
Random Forest: MAE=2.63  RMSE=3.35  R²=0.865


In [5]:
import pandas as pd

importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances.sort_values(ascending=False)

,0
RIAGENDR,0.342935
BMXBMI,0.329468
BMXWAIST,0.155727
RIDAGEYR,0.053866
BMXHT,0.052169
BMXARMC,0.017132
BMXCALF,0.016955
BMXTHICR,0.016796
BMXWT,0.014952


Training A Random Forest With Lesser Features

In [7]:
reduced_features = ['RIDAGEYR', 'RIAGENDR', 'BMXWT', 'BMXHT', 'BMXBMI', 'BMXWAIST']

X_train_r = X_train[reduced_features]
X_test_r = X_test[reduced_features]

rf_reduced = RandomForestRegressor(n_estimators=200, random_state=42)
rf_reduced.fit(X_train_r, y_train)

def evaluate(model, name, X_test_used):
    preds = model.predict(X_test_used)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f"{name}: MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.3f}")
    return preds

rf_reduced_preds = evaluate(rf_reduced, "Random Forest (reduced, 6 features)", X_test_r)

Random Forest (reduced, 6 features): MAE=2.69  RMSE=3.41  R²=0.860


In [8]:
import joblib

# Full model (9 features, including arm/calf/thigh circumference)
joblib.dump(rf, f'{base}/Models/bodyfat_model_full.joblib')

# Reduced model (6 features, no arm/calf/thigh)
joblib.dump(rf_reduced, f'{base}/Models/bodyfat_model_reduced.joblib')

print("Both models saved.")

Both models saved.
